In [1]:
#GUIA 6 - ArcGIS
import pandas as pd
from pathlib import Path

BASE = Path('/content/PROYECTO_DATOS_IA_IDS221')

carpetas = [
    'datos/geo',
    'documentos/geo',
    'evidencias/geo',
    'visualizaciones',
    'notebooks',
]

for c in carpetas:
  (BASE / c).mkdir(parents=True, exist_ok=True)

(BASE/'README.md').touch()

print("Estructura creada:----------------")
for p in sorted(BASE.rglob('*')):
  nivel = len(p.relative_to(BASE).parts) - 1
  print('  '*nivel + p.name +('/' if p.is_dir() else ''))


Estructura creada:----------------
README.md
datos/
  geo/
documentos/
  geo/
evidencias/
  geo/
notebooks/
visualizaciones/


In [3]:
#DEFINIR RUTA DEL CSV DE SUCURSALES
ruta = BASE /'datos'/'geo'/'sucursales_dataretail_guia06.csv'

#DEFINIR RUTA DEL CSV DE DEMANDA
ruta_dem = BASE/'datos'/'geo'/'demanda_dataretail_guia06.csv'

faltantes = [r for r in (ruta, ruta_dem) if not r.exists()]
if faltantes:
  raise FileNotFoundError(f"Faltan: {faltantes}")
else:
  print('Ambos archivos CSV existen y estan listos para usar')

Ambos archivos CSV existen y estan listos para usar


In [4]:
#LEER CSV DE SUCURSALES
df = pd.read_csv(ruta)

#LEER CSV DE DEMANDA
df_dem = pd.read_csv(ruta_dem)

#IMPRIMIR INFORMACIÓN DE SUCURSALES
print("\n------Archivo Sucursales -------\n")
print(f"Registros: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print(df.head())

#IMPRIMIR INFORMACIÓN DE DEMANDA
print("\n------Archivo Sucursales -------\n")
print(f"Registros: {len(df_dem)}")
print(f"Columnas: {df_dem.columns.tolist()}")
print(df_dem.head())


------Archivo Sucursales -------

Registros: 15
Columnas: ['municipio', 'departamento', 'region', 'latitud', 'longitud', 'ventas_mes', 'clientes_mes', 'ticket_promedio', 'variacion_pct', 'categoria_dominante']
      municipio  departamento  region  latitud  longitud  ventas_mes  \
0  San Salvador  San Salvador  Centro  13.6929  -89.2182       28000   
1   Santa Tecla   La Libertad  Centro  13.6769  -89.2797       24500   
2     Soyapango  San Salvador  Centro  13.7102  -89.1372       21000   
3         Apopa  San Salvador  Centro  13.8072  -89.1793       18769   
4     Mejicanos  San Salvador  Centro  13.7392  -89.2114       23311   

   clientes_mes  ticket_promedio  variacion_pct categoria_dominante  
0           973            28.78           12.0    Farmacia y salud  
1           730            33.56            7.9      Ropa y calzado  
2           606            34.65            4.0    Farmacia y salud  
3           575            32.64            8.6         Electrónica  
4     

In [5]:
#CONVERTIR COORDENADAS A NUMERO
df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')

df_dem['latitud'] = pd.to_numeric(df_dem['latitud'], errors='coerce')
df_dem['longitud'] = pd.to_numeric(df_dem['longitud'], errors='coerce')

print("Coordenadas convertidas a numerico ------------")
print("----Sucursales----")
print(df[['municipio', 'latitud', 'longitud']])


Coordenadas convertidas a numerico ------------
----Sucursales----
           municipio  latitud  longitud
0       San Salvador  13.6929  -89.2182
1        Santa Tecla  13.6769  -89.2797
2          Soyapango  13.7102  -89.1372
3              Apopa  13.8072  -89.1793
4          Mejicanos  13.7392  -89.2114
5          Santa Ana  13.9942  -89.5597
6          Sonsonate  13.7189  -89.7241
7         Ahuachapán  13.9214  -89.8451
8         San Miguel  13.4833  -88.1833
9           Usulután  13.3500  -88.4500
10          La Unión  13.3369  -87.8442
11      Zacatecoluca  13.5167  -88.8667
12       San Vicente  13.6411  -88.7847
13      Chalatenango  14.0333  -88.9333
14  Nueva Concepción  14.1167  -89.2833


In [6]:
#VALIDACION DE PLAUSIBILIDAD
mask_valida = (
    df['latitud'].between(13, 15) &
    df['longitud'].between(-91, 87)
)

print(f"Total de registros: {len(df)}")
print(f"Coordenadas validas: {mask_valida.sum()}")
print(f"Coordenadas a revisar: {(~mask_valida).sum()}")

if (~mask_valida).sum() > 0:
  print("\nCoordenadas invalidas: ----------")
  print(df[~mask_valida])
else:
  print("\nTodas las coordenadas estan dentro de El Salvador")

Total de registros: 15
Coordenadas validas: 15
Coordenadas a revisar: 0

Todas las coordenadas estan dentro de El Salvador


In [7]:
#REVISAR NULOS EN SUCURSALES
print("Nulos en sucursales ----------")
print(df.isna().sum())

#REVISAR DUPLICADOS EN SUCURSALES
print(f"\nDuplicados en sucursales: {df.duplicated().sum()}")

#REVISAR COORDENADAS REPETIDAS EN SUCURSALES
duplicados_coords = df.duplicated(subset=['latitud', 'longitud'], keep=False)
print(f"\nCoordenadas repetidas en sucursales: {duplicados_coords}")

#REVISAR NULOS EN DEMANDA
print("\nNulos en demanda ----------")
print(df_dem.isna().sum())

#REVISAR DUPLICADOS EN DEMANDA
print(f"\nDuplicados en demanda: {df_dem.duplicated().sum()}")

Nulos en sucursales ----------
municipio              0
departamento           0
region                 0
latitud                0
longitud               0
ventas_mes             0
clientes_mes           0
ticket_promedio        0
variacion_pct          0
categoria_dominante    0
dtype: int64

Duplicados en sucursales: 0

Coordenadas repetidas en sucursales: 0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
dtype: bool

Nulos en demanda ----------
id_evento               0
municipio_referencia    0
region                  0
latitud                 0
longitud                0
canal                   0
monto_estimado          0
fecha                   0
dtype: int64

Duplicados en demanda: 0


In [8]:
#EXPORTAR VERSION VALIDADA
ruta_salida = BASE/'datos'/'geo'/'sucursales_dataretail_guia06_validada.csv'
df.to_csv(ruta_salida, index=False, encoding='utf-8')

print(f"Archivo validado guardado: {ruta_salida}")

#DESCARGAR
from google.colab import files
files.download(str(ruta_salida))

Archivo validado guardado: /content/PROYECTO_DATOS_IA_IDS221/datos/geo/sucursales_dataretail_guia06_validada.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>